# audio-eval-suite — quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/#fileId=https://huggingface.co/datasets/mandipgoswami/audio-eval-suite/resolve/main/notebooks/quickstart.ipynb)

A small standardized audio-robustness eval set: synthetic speech-like probes convolved with 300 real-synthetic RIRs across 5 conditions (clean / reverb / noisy@10,5,0 dB), with ground-truth room acoustics (T60, DRR, C50, EDT) per clip. **1,500 rows.**

- Dataset: https://huggingface.co/datasets/mandipgoswami/audio-eval-suite
- Eval harness + submission format: [`eval/README.md`](https://huggingface.co/datasets/mandipgoswami/audio-eval-suite/blob/main/eval/README.md)

Two reference tasks: **(A)** reverb-level classification → accuracy, **(B)** RT60 regression → MAE.

In [ ]:
# One-cell install (Colab)
!pip install -q datasets soundfile scipy numpy

In [ ]:
from datasets import load_dataset

ds = load_dataset("mandipgoswami/audio-eval-suite", split="test")
print(ds)
ds.to_pandas().head(5)[['id','condition','reverb_bin','t60_s','drr_db','c50_db']]

In [ ]:
# Decode one audio clip
import soundfile as sf
from huggingface_hub import snapshot_download

root = snapshot_download("mandipgoswami/audio-eval-suite", repo_type="dataset")
row = ds[1]  # a 'reverb' row
wav, fs = sf.read(f"{root}/{row['audio']}")
print(row['id'], row['condition'], 'fs=', fs, 'dur=', len(wav)/fs, 's', 't60=', row['t60_s'])

In [ ]:
# Run the reference metric inline: Schroeder RT60 estimate for this clip
import numpy as np

def estimate_t60_schroeder(wav, fs):
    x = np.asarray(wav, float)
    e = x**2
    sch = np.cumsum(e[::-1])[::-1]; sch = sch / sch[0]
    db = 10*np.log10(np.maximum(sch, 1e-12))
    idx = np.where((db <= -5) & (db >= -25))[0]
    if idx.size < 10: return float('nan')
    t = idx/fs; y = db[idx]
    a = np.sum((t-t.mean())*(y-y.mean()))/np.sum((t-t.mean())**2)
    return float(-60.0/a) if a < 0 else float('nan')

est = estimate_t60_schroeder(wav, fs)
print(f"RT60  estimate={est:.3f}s   ground-truth={row['t60_s']:.3f}s")

In [ ]:
# Full baseline over the whole set (writes eval/baseline_results.json).
# Download the eval script from the repo, then run it against the Hub data:
import urllib.request, os
url = "https://huggingface.co/datasets/mandipgoswami/audio-eval-suite/resolve/main/eval/run_eval.py"
os.makedirs("eval", exist_ok=True)
urllib.request.urlretrieve(url, "eval/run_eval.py")
!python eval/run_eval.py            # prints Task A accuracy + Task B MAE